[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gnoejh/AIBookGitHub/blob/main/15_invocation.ipynb)


# OpenAI Agents SDK in the Invocation Layer

The OpenAI Agents SDK can serve as a provider-agnostic invocation substrate with built-in tool execution and light orchestration. In this stack, treat it primarily as:
- Invocation (2): abstracts model calls and routes via configured providers (optionally through LiteLLM).
- Capability Integration (5): enables tool/function registration and safe execution.
- Optional light Coordination (4): threads, steps, and event hooks can orchestrate simple flows, but complex multi-agent/task decomposition remains in Coordination.

Key mapping:
- Execution (1): raw API clients/drivers still exist beneath the SDK.
- Invocation (2): the SDK wraps/normalizes calls, handling retries and request shaping.
- Capability Integration (5): the SDK’s tool registry is the entry point for function/tool calling.

Example (illustrative, adapt to your SDK version):

```python
# Pseudo-code (do not execute as-is):
from openai_agents import Agent, Tool

# Define a tool (Python function) and register
@Tool.register(name="search_docs", description="Searches internal docs")
def search_docs(query: str) -> str:
    ...  # implement retrieval

# Create an agent with a default model and registered tools
agent = Agent(model="gpt-4o-mini", tools=[search_docs])

# Invoke the agent with instructions and optional inputs
response = agent.run("Find policies related to data retention and summarize in 3 bullets.")
print(response.output)
```

Notes:
- For multi-model routing or non-OpenAI backends, pair with LiteLLM (`openai-agents[litellm]`) to route to Anthropic, Azure OpenAI, local, etc.
- Keep business-critical orchestration in Coordination (4) for transparency and testability; leverage the SDK’s callbacks/events to integrate tracing and metrics.


# AI System Architecture: Hierarchy - API Client Layer

## Overview: Layer 2 in the Hierarchy

### Complete Hierarchy (Recap)

```
┌─────────────────────────────────────────┐
│  Layer 4: AGENTS & ORCHESTRATION        │  ← Multi-agent systems, workflows
├─────────────────────────────────────────┤
│  Layer 3: STATE MANAGEMENT              │  ← Conversation history, context
├─────────────────────────────────────────┤
│  Layer 2: API CLIENTS (THIS LAYER)      │  ← HTTP clients, abstraction, routing
├─────────────────────────────────────────┤
│  Layer 1: MODEL PROVIDERS               │  ← Where models execute (covered)
└─────────────────────────────────────────┘
```

### What We Learned in Layer 1

- **Model Providers** are where AI models physically run (OpenAI servers, Groq LPUs, your GPU)
- Providers are **stateless** - no memory between requests
- Three types: Proprietary Cloud, Open Model Cloud, Local Self-Hosted

### What is Layer 2?

**Layer 2 (API Client Layer)** is the **software that connects to providers**. It handles:
1. **HTTP communication** - Making web requests to provider APIs
2. **Authentication** - Managing API keys and tokens
3. **Request formatting** - Converting your code to provider-specific formats
4. **Error handling** - Retries, timeouts, fallbacks
5. **Abstraction** - Single interface for multiple providers

---

## 2.1 Understanding API Clients

### What is an API Client?

An **API client** is a library that:
- Wraps HTTP requests in easy-to-use functions
- Handles authentication automatically
- Provides type hints and documentation
- Manages connections, timeouts, retries

### The Simplest Example: Raw HTTP vs API Client

In [ ]:
# METHOD 1: Raw HTTP (No API client - the hard way)
import requests
import json
import os

url = "https://api.openai.com/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {os.getenv('OPENAI_API_KEY')}",
    "Content-Type": "application/json"
}
data = {
    "model": "gpt-4o-mini",
    "messages": [{"role": "user", "content": "Hello!"}]
}

response = requests.post(url, headers=headers, json=data)
result = response.json()
print("Raw HTTP Response:")
print(result['choices'][0]['message']['content'])

In [ ]:
# METHOD 2: API Client (The easy way)
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello!"}]
)

print("API Client Response:")
print(response.choices[0].message.content)

**Why use an API client?**
- ✅ Cleaner code (no manual header construction)
- ✅ Automatic retries on network errors
- ✅ Type hints for IDE autocomplete
- ✅ Better error messages
- ✅ Handles streaming, async, and edge cases

### API Client Architecture

```
Your Code
    │
    ├─ response = client.chat.completions.create(...)
    │
    ↓
API Client Library (e.g., openai package)
    │
    ├─ Builds HTTP request
    ├─ Adds authentication headers
    ├─ Serializes to JSON
    ├─ Sends via HTTPS
    │
    ↓
Provider API (e.g., api.openai.com)
    │
    ├─ Validates request
    ├─ Routes to model
    ├─ Returns response
    │
    ↓
API Client Library
    │
    ├─ Parses JSON response
    ├─ Creates Python objects
    ├─ Handles errors
    │
    ↓
Your Code receives nice Python object
```

## 2.2 Types of API Clients

### Classification by Scope

| Type | What It Connects To | Examples | Provider Count |
|------|---------------------|----------|----------------|
| **Single-Provider Client** | One specific provider | `openai`, `anthropic`, `google-generativeai` | 1 |
| **Multi-Provider Client** | Many providers via translation | `litellm`, `langchain` | 100+ |
| **Aggregator Client** | One endpoint that routes internally | OpenRouter, OpenAI SDK → OpenRouter | 200+ |

### Visual Comparison

**Single-Provider Client:**
```
Your Code → openai library → OpenAI API → GPT-4
                 ↓
            (only works with OpenAI)
```

**Multi-Provider Client (LiteLLM):**
```
Your Code → litellm library ─┬→ OpenAI API → GPT-4
                             ├→ Groq API → LLaMA
                             ├→ Anthropic API → Claude
                             └→ Local Ollama → Mistral
                 ↓
         (translates to each provider)
```

**Aggregator Client (OpenRouter):**
```
Your Code → openai library → OpenRouter API ─┬→ OpenAI
                                              ├→ Groq
                                              ├→ Anthropic
                                              └→ Google
                 ↓
         (OpenRouter routes for you)
```

## 2.3 Single-Provider Clients

### Example 1: OpenAI Client

In [ ]:
# Install: pip install openai python-dotenv
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

# Single-provider client - ONLY works with OpenAI
client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY')
)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is an API client?"}]
)

print("OpenAI Client (Single-Provider):")
print(f"Model: {response.model}")
print(f"Response: {response.choices[0].message.content}")
print(f"\nLimitation: Can ONLY call OpenAI models")

### Example 2: Anthropic Client

In [ ]:
# Install: pip install anthropic
from anthropic import Anthropic
import os
from dotenv import load_dotenv
from pathlib import Path


# Different client, different API format
# Load .env from project root
project_root = Path('.').resolve()
for _ in range(3):
    if (project_root / 'pyproject.toml').exists():
        break
    if project_root.parent == project_root:
        break
    project_root = project_root.parent

env_path = project_root / '.env'
load_dotenv(dotenv_path=env_path)

client = Anthropic(
    api_key=os.getenv('ANTHROPIC_API_KEY')
)

response = client.messages.create(
    model="claude-3-haiku-20240307",  # Free lightweight Claude model
    max_tokens=100,
    messages=[{"role": "user", "content": "What is an API client?"}]
)

print("Anthropic Client (Single-Provider):")
print(f"Model: {response.model}")
print(f"Response: {response.content[0].text}")
print(f"\nNotice: Different API structure than OpenAI!")
print(f"  OpenAI: response.choices[0].message.content")
print(f"  Anthropic: response.content[0].text")

### Example 3: Google Generative AI Client

In [ ]:
# Install: pip install google-generativeai
#
# To get a Google API key for Gemini:
# Option 1 (EASIEST - Recommended):
#   1. Go to https://aistudio.google.com/app/apikey
#   2. Sign in with your Google account
#   3. Click "Create API Key"
#   5. Free tier: 60 requests/minute, generous free quota
#
# Option 2 (Google Cloud Console):
#   1. Go to https://console.cloud.google.com/
#   2. Create a new project
#   3. Enable "Generative Language API"
#   4. Create credentials > API Key
#
try:
    import google.generativeai as genai
    import os
    from dotenv import load_dotenv
    from pathlib import Path

    # Load .env from project root
    project_root = Path('.').resolve()
    for _ in range(3):
        if (project_root / 'pyproject.toml').exists():
            break
        if project_root.parent == project_root:
            break
        project_root = project_root.parent

    env_path = project_root / '.env'
    load_dotenv(dotenv_path=env_path)

    # Try GOOGLE_API_KEY first, fallback to GOOGLE_GEMINI_API_KEY
    api_key = os.getenv('GOOGLE_API_KEY') or os.getenv('GOOGLE_GEMINI_API_KEY')
    if not api_key:
        print("❌ GOOGLE_API_KEY or GOOGLE_GEMINI_API_KEY not found in .env file")
        print("\nTo get a free API key:")
        print("  1. Visit: https://aistudio.google.com/app/apikey")
        print("  2. Sign in with your Google account")
        print("  3. Click 'Create API Key'")
        print("\nFree tier: 60 requests/minute, generous free quota")
    else:
        # Yet another different API format
        genai.configure(api_key=api_key)
    # Using gemini-1.5-flash (better free tier availability than gemini-2.0-flash-exp)
        # Using gemini-1.5-flash (better free tier availability than gemini-2.0-flash-exp)
        model = genai.GenerativeModel('gemini-1.5-flash')

        response = model.generate_content("What is an API client?")

        print("Google Client (Single-Provider):")
        print(f"Model: gemini-1.5-flash")
        print(f"Response: {response.text}")
        print(f"\nNotice: Completely different API!")
        print(f"  OpenAI: client.chat.completions.create(...)")
        print(f"  Anthropic: client.messages.create(...)")
        print(f"  Google: model.generate_content(...)")
except ImportError:
    print("❌ google-generativeai not installed.")
    print("   Install with: pip install google-generativeai")
    print("   Or add to requirements.txt: google-generativeai>=0.3.0")
except Exception as e:
    error_msg = str(e)
    if "429" in error_msg or "quota" in error_msg.lower() or "rate" in error_msg.lower():
        print("⚠️  Rate Limit / Quota Exceeded")
        print("\nThis means your API key is working, but you've hit the free tier limits.")
        print("\nSolutions:")
        print("  1. Wait a few minutes and try again (free tier: 60 requests/minute)")
        print("  2. Check your usage: https://ai.dev/usage?tab=rate-limit")
        print("  3. The model gemini-2.0-flash-exp may not be available on free tier")
        print("  4. Try using gemini-1.5-flash or gemini-pro instead")
    else:
        print(f"❌ Error: {e}")
        print("\nTroubleshooting:")
        print("  1. Make sure GOOGLE_API_KEY or GOOGLE_GEMINI_API_KEY is set in your .env file")
        print("  2. Get a free key at: https://aistudio.google.com/app/apikey")
        print("  3. Check that the API key is valid and not expired")
    print(f"❌ Error: {e}")
    print("\nTroubleshooting:")
    print("  1. Make sure GOOGLE_API_KEY or GOOGLE_GEMINI_API_KEY is set in your .env file")
    print("  2. Get a free key at: https://aistudio.google.com/app/apikey")
    print("  3. Check that the API key is valid and not expired")


### Problem with Single-Provider Clients

**To support multiple providers, you need different code:**

```python
# OpenAI
openai_response = openai_client.chat.completions.create(
    model="gpt-4",
    messages=[...]
)
text = openai_response.choices[0].message.content

# Anthropic - DIFFERENT code structure
anthropic_response = anthropic_client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=100,
    messages=[...]
)
text = anthropic_response.content[0].text

# Google - COMPLETELY DIFFERENT
google_response = google_model.generate_content("...")
text = google_response.text
```

**Solution:** Multi-provider clients that abstract these differences.

## 2.4 Multi-Provider Clients: LiteLLM

### What is LiteLLM?

**LiteLLM** is a **translation layer** that:
1. Accepts OpenAI-format requests
2. Translates them to each provider's specific format
3. Returns responses in a consistent format

### LiteLLM Architecture

```
Your Code (OpenAI format)
    │
    ├─ litellm.completion(model="groq/llama", messages=[...])
    │
    ↓
LiteLLM Library
    │
    ├─ Detects provider from model name ("groq/...")
    ├─ Translates OpenAI format → Groq format
    ├─ Adds Groq-specific authentication
    ├─ Calls Groq API
    │
    ↓
Provider API (Groq in this case)
    │
    ├─ Returns response in Groq format
    │
    ↓
LiteLLM Library
    │
    ├─ Translates Groq format → OpenAI format
    ├─ Returns standardized response
    │
    ↓
Your Code (receives OpenAI-compatible object)
```

### Key Feature: Provider Prefix

LiteLLM uses **provider prefixes** in model names:

```python
"openai/gpt-4"              → Calls OpenAI
"anthropic/claude-3-opus"   → Calls Anthropic
"groq/llama-3.1-8b-instant"        → Calls Groq
"gemini/gemini-2.0-flash"   → Calls Google
"ollama/llama3.1"           → Calls local Ollama
"huggingface/meta-llama/..." → Calls HuggingFace
```

## 2.5 LiteLLM Code Examples

In [ ]:
# Install: pip install litellm
from litellm import completion
from dotenv import load_dotenv
from pathlib import Path
import os

# Load .env from project root
project_root = Path('.').resolve()
for _ in range(3):
    if (project_root / 'pyproject.toml').exists():
        break
    if project_root.parent == project_root:
        break
    project_root = project_root.parent

env_path = project_root / '.env'
load_dotenv(dotenv_path=env_path)


# Same code structure, different providers!

# 1. OpenAI (via LiteLLM)
response = completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Count to 3"}],
    api_key=os.getenv('OPENAI_API_KEY')
)
print("OpenAI via LiteLLM:")
print(response.choices[0].message.content)

# 2. Groq (via LiteLLM)
response = completion(
    model="groq/llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Count to 3"}],
    api_key=os.getenv('GROQ_API_KEY')
)
print("\nGroq via LiteLLM:")
print(response.choices[0].message.content)

# 3. Ollama (via LiteLLM)
response = completion(
    model="ollama/llama3.1",
    messages=[{"role": "user", "content": "Count to 3"}],
    api_base="http://localhost:11434"
)
print("\nOllama via LiteLLM:")
print(response.choices[0].message.content)

print("\n✅ Same code structure for all providers!")

### LiteLLM Advanced Features

#### Feature 1: Automatic Fallbacks

In [ ]:
from litellm import completion
from dotenv import load_dotenv
from pathlib import Path
import os

# Load .env from project root
project_root = Path('.').resolve()
for _ in range(3):
    if (project_root / 'pyproject.toml').exists():
        break
    if project_root.parent == project_root:
        break
    project_root = project_root.parent

env_path = project_root / '.env'
load_dotenv(dotenv_path=env_path)


# Try Groq first, fallback to OpenAI if it fails
response = completion(
    model="groq/llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Hello"}],
    api_key=os.getenv('GROQ_API_KEY'),
    fallbacks=["openai/gpt-4o-mini"]  # Automatic fallback
)

print("Used model:", response.model)
print("Response:", response.choices[0].message.content)
print("\n💡 If Groq fails (rate limit, error), automatically tries OpenAI")

#### Feature 2: Cost Tracking

In [ ]:
import litellm
import os
from dotenv import load_dotenv
from pathlib import Path

# Load .env from project root
project_root = Path('.').resolve()
for _ in range(3):
    if (project_root / 'pyproject.toml').exists():
        break
    if project_root.parent == project_root:
        break
    project_root = project_root.parent

env_path = project_root / '.env'
load_dotenv(dotenv_path=env_path)

# Make multiple calls and track costs
total_cost = 0

for i in range(3):
    response = litellm.completion(
        model="openai/gpt-4o-mini",
        messages=[{"role": "user", "content": f"Message {i+1}"}],
        api_key=os.getenv('OPENAI_API_KEY')
    )
    
    # Calculate cost for this call
    cost = litellm.completion_cost(completion_response=response)
    total_cost += cost
    
    print(f"Call {i+1}: ${cost:.6f}")

print(f"\nTotal cost: ${total_cost:.6f}")
print("\n💡 LiteLLM tracks costs automatically based on token usage")

#### Feature 3: Load Balancing Router

In [ ]:
from litellm import Router
import os
from dotenv import load_dotenv
from pathlib import Path

# Load .env from project root
project_root = Path('.').resolve()
for _ in range(3):
    if (project_root / 'pyproject.toml').exists():
        break
    if project_root.parent == project_root:
        break
    project_root = project_root.parent

env_path = project_root / '.env'
load_dotenv(dotenv_path=env_path)

# Configure multiple providers for the same logical "model"
router = Router(
    model_list=[
        {
            "model_name": "llama-fast",  # Logical name
            "litellm_params": {
                "model": "groq/llama-3.1-8b-instant",
                "api_key": os.getenv('GROQ_API_KEY')
            }
        },
        {
            "model_name": "llama-fast",  # Same logical name
            "litellm_params": {
                "model": "ollama/llama3.1",
                "api_base": "http://localhost:11434"
            }
        }
    ]
)

# Router automatically distributes load
for i in range(4):
    response = router.completion(
        model="llama-fast",
        messages=[{"role": "user", "content": f"Request {i+1}"}]
    )
    print(f"Request {i+1} → {response.model}")

print("\n💡 Router distributed requests across Groq and Ollama")

## 2.6 Aggregator Clients: OpenRouter

### What is OpenRouter?

**OpenRouter** is a **hosted aggregator service** that:
1. Provides a single API endpoint
2. Routes your requests to 200+ providers internally
3. You only need ONE API key

### OpenRouter vs LiteLLM

| Aspect | LiteLLM | OpenRouter |
|--------|---------|------------|
| **Type** | Code library | Hosted service |
| **Where translation happens** | Your code | OpenRouter servers |
| **API keys needed** | One per provider | Just OpenRouter key |
| **Request flow** | You → Provider directly | You → OpenRouter → Provider |
| **Cost** | Provider pricing | Provider pricing + markup |
| **Rate limits** | Provider limits | OpenRouter limits (20/min free) |
| **Fallbacks** | You configure | OpenRouter handles |

### OpenRouter Architecture

```
Your Code (OpenAI-compatible format)
    │
    ├─ POST https://openrouter.ai/api/v1/chat/completions
    │  Authorization: Bearer YOUR_OPENROUTER_KEY
    "google/gemini-1.5-flash:free",  # More stable free model
    │
    ↓
OpenRouter Service
    │
    ├─ Authenticates your request
    ├─ Parses model identifier ("google/...")
    ├─ Routes to Google's API
    ├─ Handles Google authentication
    ├─ Translates formats if needed
    │
    ↓
Google Gemini API
    │
    ├─ Executes model
    ├─ Returns response
    │
    ↓
OpenRouter Service
    │
    ├─ Standardizes response
    ├─ Adds usage tracking
    ├─ Returns to you
    │
    ↓
Your Code (OpenAI-compatible response)
```

### OpenRouter Code Example

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path
import os

# Load .env from project root
project_root = Path('.').resolve()
for _ in range(3):
    if (project_root / 'pyproject.toml').exists():
        break
    if project_root.parent == project_root:
        break
    project_root = project_root.parent

env_path = project_root / '.env'
load_dotenv(dotenv_path=env_path)


# Use OpenAI client, but point to OpenRouter
client = OpenAI(
    api_key=os.getenv('OPENROUTER_API_KEY'),
    base_url='https://openrouter.ai/api/v1'
)

# Access ANY model through OpenRouter
models_to_try = [
    # Note: For paid accounts, remove :free suffix
    # For free tier, keep :free suffix
    "google/gemini-1.5-flash:free",  # Free tier
    # "google/gemini-1.5-flash",  # Paid account (uncomment if you have paid account)
    "meta-llama/llama-3.1-8b-instruct:free",  # Free tier
    # "meta-llama/llama-3.1-8b-instruct",  # Paid account (uncomment if you have paid account)
    "mistralai/mistral-7b-instruct:free"
]

for model in models_to_try:
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Say hello in 3 words"}],
            max_tokens=20
        )
        print(f"{model}:")
        print(f"  → {response.choices[0].message.content}")
    except Exception as e:
        error_msg = str(e)
        if "429" in error_msg or "rate" in error_msg.lower() or "quota" in error_msg.lower():
            print(f"{model}: ⚠️  Rate limited (free tier: 20 requests/min)")
            print("   Tip: Wait a minute or add your own API key at https://openrouter.ai/settings/integrations")
        else:
            print(f"{model}: Error - {error_msg[:100]}...")
        error_msg = str(e)
        if "429" in error_msg or "rate" in error_msg.lower() or "quota" in error_msg.lower():
            print(f"{model}: ⚠️  Rate limited (free tier: 20 requests/min)")
            print("   Tip: Wait a minute or add your own API key at https://openrouter.ai/settings/integrations")
        else:
            print(f"{model}: Error - {error_msg[:100]}...")

print("\n✅ All models accessed with ONE API key!")

In [9]:
from openai import OpenAI
from litellm import completion
import os
from dotenv import load_dotenv
from pathlib import Path

# Load .env from project root
project_root = Path('.').resolve()
for _ in range(3):
    if (project_root / 'pyproject.toml').exists():
        break
    if project_root.parent == project_root:
        break
    project_root = project_root.parent

env_path = project_root / '.env'
load_dotenv(dotenv_path=env_path)

prompt = "What is 2+2?"

# APPROACH 1: Single-Provider (OpenAI)
print("=" * 60)
print("APPROACH 1: Single-Provider Client (OpenAI)")
print("=" * 60)
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
response = openai_client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": prompt}]
)
print(f"Response: {response.choices[0].message.content}")
print(f"Pros: Simple, direct access")
print(f"Cons: Only works with OpenAI")

# APPROACH 2: Multi-Provider (LiteLLM)
print("\n" + "=" * 60)
print("APPROACH 2: Multi-Provider Client (LiteLLM)")
print("=" * 60)
response = completion(
    model="groq/llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}],
    api_key=os.getenv('GROQ_API_KEY')
)
print(f"Response: {response.choices[0].message.content}")
print(f"Pros: Works with 100+ providers, direct access")
print(f"Cons: Need API key per provider")

# APPROACH 3: Aggregator (OpenRouter)
print("\n" + "=" * 60)
print("APPROACH 3: Aggregator Client (OpenRouter)")
print("=" * 60)
openrouter_client = OpenAI(
    api_key=os.getenv('OPENROUTER_API_KEY'),
    base_url='https://openrouter.ai/api/v1'
)
try:
    response = openrouter_client.chat.completions.create(
        model="openai/gpt-3.5-turbo",  # Most reliable model for paid accounts
        # Alternative models if needed:
        # model="meta-llama/llama-3.1-8b-instruct",
        # model="google/gemini-1.5-flash",
        messages=[{"role": "user", "content": prompt}]
    )
    print(f"Response: {response.choices[0].message.content}")
    print(f"Pros: One key for 200+ models")
    print(f"Cons: Rate limits, added latency")
except Exception as e:
    error_msg = str(e)
    print(f"❌ OpenRouter Error: {error_msg[:200]}")
    print("\nTroubleshooting:")
    print("  1. Check API key: OPENROUTER_API_KEY in .env file")
    print("  2. Verify model access: https://openrouter.ai/settings/keys")
    print("  3. For paid accounts, use model names WITHOUT :free suffix")
    print("  4. Try: openai/gpt-3.5-turbo (most reliable)")
    print("  5. Check status: https://status.openrouter.ai")


APPROACH 1: Single-Provider Client (OpenAI)
Response: 2+2 equals 4.
Pros: Simple, direct access
Cons: Only works with OpenAI

APPROACH 2: Multi-Provider Client (LiteLLM)
Response: 2 + 2 = 4.
Pros: Works with 100+ providers, direct access
Cons: Need API key per provider

APPROACH 3: Aggregator Client (OpenRouter)
Response: 2+2 equals 4.
Pros: One key for 200+ models
Cons: Rate limits, added latency


## 2.8 Understanding Async vs Sync Clients

### What is Async?

**Synchronous (blocking):**
```python
response1 = client.chat.completions.create(...)  # Wait for response
response2 = client.chat.completions.create(...)  # Then make next call
# Total time: 2 seconds (1s each)
```

**Asynchronous (non-blocking):**
```python
response1 = await client.chat.completions.create(...)  # Start call 1
response2 = await client.chat.completions.create(...)  # Start call 2 (in parallel)
# Total time: 1 second (both run simultaneously)
```

### Sync vs Async Example

In [11]:
import time
import asyncio
from openai import OpenAI, AsyncOpenAI
import os

# SYNCHRONOUS - One after another
def sync_example():
    client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    
    start = time.time()
    
    response1 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Count to 3"}]
    )
    
    response2 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Count to 3"}]
    )
    
    elapsed = time.time() - start
    print(f"Synchronous: {elapsed:.2f} seconds")
    return elapsed

# ASYNCHRONOUS - Parallel
async def async_example():
    client = AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    
    start = time.time()
    
    # Both calls start simultaneously
    task1 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Count to 3"}]
    )
    
    task2 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Count to 3"}]
    )
    
    # Wait for both to complete
    response1, response2 = await asyncio.gather(task1, task2)
    
    elapsed = time.time() - start
    print(f"Asynchronous: {elapsed:.2f} seconds")
    return elapsed

# Run both
sync_time = sync_example()
async_time = await async_example()

print(f"\n⚡ Async was {sync_time/async_time:.1f}x faster!")
print("\n💡 Use AsyncOpenAI for multi-agent systems")

Synchronous: 2.20 seconds
Asynchronous: 1.90 seconds

⚡ Async was 1.2x faster!

💡 Use AsyncOpenAI for multi-agent systems


## 2.9 Key Takeaways - API Client Layer

### Essential Concepts

1. **API Clients Wrap HTTP Communication**
   - Convert your code into HTTP requests
   - Handle authentication, retries, errors
   - Provide cleaner interface than raw HTTP

2. **Three Types of Clients**
   - **Single-provider**: `openai`, `anthropic`, `google-generativeai` (1 provider)
   - **Multi-provider**: `litellm` (100+ providers via translation)
   - **Aggregator**: OpenRouter (200+ providers via routing)

3. **Each Provider Has Different API Format**
   - OpenAI: `response.choices[0].message.content`
   - Anthropic: `response.content[0].text`
   - Google: `response.text`
   - **Solution**: LiteLLM or OpenRouter standardize these

4. **LiteLLM vs OpenRouter**
   - **LiteLLM**: Library in your code, direct provider access, need multiple keys
   - **OpenRouter**: Hosted service, one key for all, has rate limits/markup

5. **Async is Critical for Multi-Agent Systems**
   - `AsyncOpenAI` allows parallel requests
   - 2x-10x faster for multiple agents
   - Required for real-time conversations

6. **The Client Layer is Still Stateless**
   - API clients just make requests
   - Don't manage conversation history
   - State management is Layer 3 (next notebook)

### Architecture Summary

```
Layer 2: API CLIENT LAYER (What we learned)
├─ Single-Provider Clients
│  ├─ openai package → OpenAI only
│  ├─ anthropic package → Anthropic only
│  └─ google-generativeai → Google only
│
├─ Multi-Provider Clients
│  └─ litellm → Translates to 100+ providers
│     ├─ Direct provider access
│     ├─ Fallbacks & routing
│     └─ Cost tracking
│
└─ Aggregator Clients
   └─ OpenRouter → Routes to 200+ providers
      ├─ One API key
      ├─ Automatic routing
      └─ Rate limits apply
```

### Next: Layer 3 - State Management

In the next notebook, we'll cover:
- Managing conversation history
- Building stateful agents
- Context windows and memory
- Your `ConversationAgent` class (what you built!)

## Practice Exercises

### Exercise 1: Convert Between Client Types

Take this OpenAI code and convert it to:
1. LiteLLM calling Groq
2. OpenRouter calling Google Gemini

```python
from openai import OpenAI

client = OpenAI(api_key="...")
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello"}]
)
print(response.choices[0].message.content)
```

### Exercise 2: Build a Fallback System

Using LiteLLM, create a function that:
1. Tries Groq first (fastest)
2. Falls back to OpenAI if Groq fails
3. Falls back to Ollama (local) as last resort

### Exercise 3: Async Performance Test

Measure the speedup of async vs sync for:
- 2 parallel requests
- 5 parallel requests
- 10 parallel requests

Plot the results.

### Exercise 4: Cost Calculator

Using LiteLLM's cost tracking, calculate the monthly cost for:
- 1000 messages/day to GPT-4o-mini
- Same traffic to Groq LLaMA
- Savings by using Groq instead

---

**Next Notebook:** `3_hierarchy_state_layer.ipynb` - State Management & Conversation History